In [1]:
import xarray as xr
from dask.distributed import Client
from dask_jobqueue import PBSCluster
import matplotlib.pyplot as plt
import numpy as np

In [2]:
cluster = PBSCluster(
    cores=1,
    memory='40GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='4:00:00'
)
cluster.scale(jobs=5)
client = Client(cluster)
client

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37199 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/37199/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/37199/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.175:39625,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/37199/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [4]:
path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/'
mfd = xr.open_dataset(path+'MFD.nc', chunks={})
mfd

<xarray.Dataset> Size: 30GB
Dimensions:           (time: 753888, latitude: 82, longitude: 121)
Coordinates:
  * time              (time) datetime64[ns] 6MB 1940-01-01 ... 2025-12-31T23:...
  * latitude          (latitude) float64 656B 7.75 8.0 8.25 ... 27.5 27.75 28.0
  * longitude         (longitude) float64 968B -89.0 -88.75 ... -59.25 -59.0
Data variables:
    field_divergence  (time, latitude, longitude) float32 30GB dask.array<chunksize=(753888, 82, 121), meta=np.ndarray>

In [5]:
hi_ds = xr.open_dataset(path+'hourly_HI.nc')
hi_ds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    HI_hourly  (time, latitude, longitude) float32 30GB ...

In [6]:
years = np.arange(1940, 2026, 1)
hi_ds = hi_ds.sel(time=hi_ds.time.dt.year.isin(years))
mfd = mfd.sel(time=mfd.time.dt.year.isin(years))

In [9]:
mfd = mfd.chunk({'time': (mfd.sizes['time'] / 24), 'latitude': -1, 'longitude': -1})
hi_ds = hi_ds.chunk({'time': (hi_ds.sizes['time'] / 24), 'latitude': -1, 'longitude': -1})

In [10]:
# make hour dim lazily
hi_ds_coarse = hi_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
mfd_ds_coarse = mfd['field_divergence'].coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))

# get hour where max happens
hidmax_idx = hi_ds_coarse['HI_hourly'].argmax(dim='hour')

def sel_at_idx(ds, idx_ds):
    # add hour dimension to indexing array
    idx_ds = np.expand_dims(idx_ds, axis=-1)
    # select along hour axis and remove hour axis
    return np.take_along_axis(ds, idx_ds, axis=-1).squeeze(axis=-1)

In [11]:
mfd_peakHI = xr.apply_ufunc(sel_at_idx, mfd_ds_coarse, hidmax_idx,
                              input_core_dims=[['hour'], []],
                              output_core_dims=[[]],
                              dask='parallelized',
                              output_dtypes=[mfd_ds_coarse.dtype]
                             )
mfd_peakHI = mfd_peakHI.rename('MFD_during_HIdmax')
mfd_peakHI

<xarray.DataArray 'MFD_during_HIdmax' (day: 31412, latitude: 82, longitude: 121)> Size: 1GB
dask.array<transpose, shape=(31412, 82, 121), dtype=float32, chunksize=(1308, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [12]:
mfd_peakHI = mfd_peakHI.chunk(-1)
mfd_peakHI

<xarray.DataArray 'MFD_during_HIdmax' (day: 31412, latitude: 82, longitude: 121)> Size: 1GB
dask.array<rechunk-merge, shape=(31412, 82, 121), dtype=float32, chunksize=(31412, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [13]:
mfd_peakHI.to_netcdf(path+'MFD_HIdmax.nc')

Task exception was never retrieved
future: <Task finished name='Task-123894' coro=<Client._gather.<locals>.wait() done, defined at /glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:2367> exception=AllExit()>
Traceback (most recent call last):
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py", line 2376, in wait
    raise AllExit()
distributed.client.AllExit

KeyboardInterrupt


KeyboardInterrupt



In [ ]:
client.shutdown